In [ ]:
"""
icosahedron_cube_root.py
========================
Does 2^(-1/3) emerge from icosahedral geometry?

Target: cos(θ) = 2^(-1/3) = 0.7937005259, θ = 37.467311°

Tests:
1. All central angles between vertex pairs
2. Vertex-to-face-center angles (3-fold axes)
3. Vertex-to-edge-midpoint angles (2-fold axes)
4. Axis-to-axis angles (3-5, 3-2, 5-2)
5. Inscribed cubes: edge ratio
6. Direct ratio checks against 2^(±1/3)
"""

import numpy as np
from itertools import combinations, product

phi = (1 + np.sqrt(5)) / 2
target_cos = 2 ** (-1/3)
target_angle = np.degrees(np.arccos(target_cos))

print("=" * 70)
print("ICOSAHEDRAL CONNECTION TO 2^(-1/3)")
print("=" * 70)
print(f"Target: 2^(-1/3) = {target_cos:.10f}")
print(f"Target angle:     {target_angle:.6f}°")
print()

# Build icosahedron
verts = []
for s1, s2 in product([1, -1], repeat=2):
    verts.append([0, s1, s2*phi])
    verts.append([s1, s2*phi, 0])
    verts.append([s2*phi, 0, s1])

unique = []
for v in verts:
    if not any(np.allclose(v, u) for u in unique):
        unique.append(v)
verts = np.array(unique)
R = np.linalg.norm(verts[0])

edge_len = min(np.linalg.norm(verts[i] - verts[j])
               for i, j in combinations(range(12), 2))

print(f"Icosahedron: R = {R:.6f}, edge = {edge_len:.6f}")
print(f"edge/R = {edge_len/R:.8f}")
print(f"(edge/R)^3 = {(edge_len/R)**3:.8f}")
print(f"R/edge = {R/edge_len:.8f}")
print(f"(R/edge)^3 = {(R/edge_len)**3:.8f}")
print()

# Find faces
faces = []
for i, j, k in combinations(range(12), 3):
    dij = np.linalg.norm(verts[i] - verts[j])
    djk = np.linalg.norm(verts[j] - verts[k])
    dki = np.linalg.norm(verts[k] - verts[i])
    if abs(dij - edge_len) < 1e-8 and abs(djk - edge_len) < 1e-8 and abs(dki - edge_len) < 1e-8:
        faces.append((i, j, k))

face_centers = np.array([sum(verts[i] for i in f)/3 for f in faces])

# Edge midpoints
edge_midpoints = []
for i, j in combinations(range(12), 2):
    if abs(np.linalg.norm(verts[i] - verts[j]) - edge_len) < 1e-8:
        edge_midpoints.append((verts[i] + verts[j])/2)
edge_midpoints = np.array(edge_midpoints)

# Axes
axes_5fold = []  # through vertices
for v in verts:
    if not any(np.allclose(v, u) or np.allclose(-v, u) for u in axes_5fold):
        axes_5fold.append(v)
axes_3fold = []  # through face centers
for fc in face_centers:
    if not any(np.allclose(fc, u) or np.allclose(-fc, u) for u in axes_3fold):
        axes_3fold.append(fc)
axes_2fold = []  # through edge midpoints
for em in edge_midpoints:
    if not any(np.allclose(em, u) or np.allclose(-em, u) for u in axes_2fold):
        axes_2fold.append(em)

print(f"Axes: 5-fold = {len(axes_5fold)}, 3-fold = {len(axes_3fold)}, "
      f"2-fold = {len(axes_2fold)}")
print()

def ang(u, v):
    c = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return np.degrees(np.arccos(np.clip(c, -1, 1)))

def check(label, angles):
    """Report angles, mark matches."""
    angles = sorted(set(round(a, 5) for a in angles))
    matches = [a for a in angles if abs(a - target_angle) < 0.05]
    print(f"{label}:")
    for a in angles:
        m = "  <<<<<< MATCH" if abs(a - target_angle) < 0.05 else ""
        print(f"  {a:10.5f}°{m}")
    print()
    return matches

# ============================================================
# TEST 1: Vertex-vertex angles
# ============================================================
vv = []
for i, j in combinations(range(12), 2):
    vv.append(ang(verts[i], verts[j]))
m1 = check("Vertex-vertex central angles", vv)

# ============================================================
# TEST 2: Vertex to face center (3-fold axis)
# ============================================================
vfc = []
for v in verts:
    for fc in face_centers:
        vfc.append(ang(v, fc))
m2 = check("Vertex to face-center angles", vfc)

# ============================================================
# TEST 3: Vertex to edge midpoint (2-fold axis)
# ============================================================
vem = []
for v in verts:
    for em in edge_midpoints:
        vem.append(ang(v, em))
m3 = check("Vertex to edge-midpoint angles", vem)

# ============================================================
# TEST 4: Axis-to-axis angles
# ============================================================
a35 = []
for a3 in axes_3fold:
    for a5 in axes_5fold:
        a35.append(ang(a3, a5))
m4 = check("3-fold to 5-fold axis angles", a35)

a32 = []
for a3 in axes_3fold:
    for a2 in axes_2fold:
        a32.append(ang(a3, a2))
m5 = check("3-fold to 2-fold axis angles", a32)

a52 = []
for a5 in axes_5fold:
    for a2 in axes_2fold:
        a52.append(ang(a5, a2))
m6 = check("5-fold to 2-fold axis angles", a52)

# ============================================================
# TEST 5: Inscribed cubes
# ============================================================
print("=" * 70)
print("INSCRIBED CUBES")
print("=" * 70)

def is_cube(indices, tol=1e-5):
    sub = verts[list(indices)]
    dists = np.zeros((8, 8))
    for i, j in combinations(range(8), 2):
        d = np.linalg.norm(sub[i] - sub[j])
        dists[i, j] = d
        dists[j, i] = d
    for i in range(8):
        ds = sorted(dists[i][dists[i] > tol])
        if len(ds) != 7:
            return None
        d = ds[0]
        if not (abs(ds[0]-d)<tol and abs(ds[1]-d)<tol and abs(ds[2]-d)<tol):
            return None
        if not (abs(ds[3]-d*np.sqrt(2))<tol*3 and
                abs(ds[4]-d*np.sqrt(2))<tol*3 and
                abs(ds[5]-d*np.sqrt(2))<tol*3):
            return None
        if abs(ds[6]-d*np.sqrt(3))>tol*3:
            return None
    return d

cubes = []
for indices in combinations(range(12), 8):
    d = is_cube(indices)
    if d is not None:
        cubes.append((indices, d))

print(f"Cubes found: {len(cubes)}")
for indices, d in cubes:
    ratio = d / edge_len
    print(f"  Vertices: {indices}")
    print(f"    cube_edge = {d:.6f}")
    print(f"    cube_edge / ico_edge = {ratio:.6f}")
    print(f"    (cube_edge/ico_edge)^3 = {ratio**3:.6f}")
    if abs(ratio - target_cos) < 1e-3:
        print(f"    <<< MATCH 2^(-1/3)")
    if abs(ratio**3 - 2) < 1e-3:
        print(f"    <<< CUBE = 2")
print()

# ============================================================
# TEST 6: Direct ratio checks
# ============================================================
print("=" * 70)
print("DIRECT RATIO CHECKS")
print("=" * 70)
print()

checks = {
    "R / edge":       R / edge_len,
    "edge / R":       edge_len / R,
    "1/φ":            1/phi,
    "1/φ²":           1/phi**2,
    "√5 - 2":         np.sqrt(5) - 2,
    "√3/2":           np.sqrt(3)/2,
    "1/√3":           1/np.sqrt(3),
    "2/√3":           2/np.sqrt(3),
    "√2 - 1":         np.sqrt(2) - 1,
    "1/√2":           1/np.sqrt(2),
    "cos(37.5°)":     np.cos(np.radians(37.5)),
    "cos(37.4673°)":  target_cos,
    "2^(-1/3)":       target_cos,
    "2^(-1/2)":       2 ** -0.5,
    "2^(1/3)/2":      2 ** (1/3) / 2,
}

print(f"{'name':25s} {'value':>12s} {'^3':>12s} {'match?':>20s}")
print("-" * 70)
for name, val in checks.items():
    v3 = val**3
    match = ""
    if abs(val - target_cos) < 1e-4:
        match = "≈ 2^(-1/3)"
    if abs(v3 - 2) < 1e-3:
        match = "CUBE = 2"
    print(f"{name:25s} {val:>12.7f} {v3:>12.7f} {match:>20s}")

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)
all_matches = m1 + m2 + m3 + m4 + m5 + m6
if all_matches:
    print(f"Angle matches found: {len(all_matches)}")
else:
    print("No natural icosahedral angle equals 37.4673°.")
    print()
    print("The target does NOT appear as a direct angle in the icosahedron.")

ICOSAHEDRAL CONNECTION TO 2^(-1/3)
Target: 2^(-1/3) = 0.7937005260
Target angle:     37.467311°

Icosahedron: R = 1.902113, edge = 2.000000
edge/R = 1.05146222
(edge/R)^3 = 1.16246804
R/edge = 0.95105652
(R/edge)^3 = 0.86023870

Axes: 5-fold = 6, 3-fold = 10, 2-fold = 15

Vertex-vertex central angles:
    63.43495°
   116.56505°
   180.00000°

Vertex to face-center angles:
    37.37737°
    79.18768°
   100.81232°
   142.62263°

Vertex to edge-midpoint angles:
    31.71747°
    58.28253°
    90.00000°
   121.71747°
   148.28253°

3-fold to 5-fold axis angles:
    37.37737°
    79.18768°
   100.81232°
   142.62263°

3-fold to 2-fold axis angles:
    20.90516°
    54.73561°
    69.09484°
    90.00000°
   110.90516°
   125.26439°
   159.09484°

5-fold to 2-fold axis angles:
    31.71747°
    58.28253°
    90.00000°
   121.71747°
   148.28253°

INSCRIBED CUBES
Cubes found: 0

DIRECT RATIO CHECKS

name                             value           ^3               match?
---------------------